<a href="https://colab.research.google.com/github/CharlieHubbard7/NFLResearch/blob/main/NFL_Research_(5_26_26).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#NFL Research

This notebook contains my work for the first week of work with professor Ron Yurko at Carnegie Mellon's Sports Analytics Center.

In [ ]:
# pip install nflreadpy #This is from the NFLverse package: https://github.com/nflverse/nflreadpy
# !pip install rapidfuzz -q #This helps match the NFL roster with the madden data

In [45]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns
import nflreadpy as nfl
from rapidfuzz import fuzz, process
pd.set_option('display.max_columns', 250)
pd.set_option('display.min_rows', 10)

## NFL Verse Stats

In [4]:
pbp = nfl.load_pbp().to_pandas() # play-by-play data
player_stats = nfl.load_player_stats().to_pandas() # player game or season statistics
team_stats = nfl.load_team_stats().to_pandas() # team game or season statistics
# schedules = nfl.load_schedules().to_pandas() # game schedules and results
players = nfl.load_players().to_pandas() # player information
rosters = nfl.load_rosters().to_pandas() # team rosters
rosters_weekly = nfl.load_rosters_weekly().to_pandas() # team rosters by season-week
snap_counts = nfl.load_snap_counts().to_pandas() # snap counts
nextgen_stats = nfl.load_nextgen_stats().to_pandas() # advanced stats from nextgenstats.nfl.com
ftn_charting = nfl.load_ftn_charting().to_pandas() # charted stats from ftnfantasy.com/data
participation = nfl.load_participation().to_pandas() # participation data (historical)
draft_picks = nfl.load_draft_picks().to_pandas() # nfl draft picks
injuries = nfl.load_injuries().to_pandas() # injury statuses and practice participation
contracts = nfl.load_contracts().to_pandas() # historical contract data from OTC
officials = nfl.load_officials().to_pandas() # officials for each game
combine = nfl.load_combine().to_pandas() # nfl combine results
depth_charts = nfl.load_depth_charts().to_pandas() # depth charts
trades = nfl.load_trades().to_pandas() # trades
ff_playerids = nfl.load_ff_playerids().to_pandas() # ffverse/dynastyprocess player ids
ff_rankings = nfl.load_ff_rankings().to_pandas() # fantasypros rankings
ff_opportunity = nfl.load_ff_opportunity().to_pandas() # expected yards, touchdowns, and fantasy points

##Madden Stats

In [59]:
madden2026 = pd.read_csv("/content/drive/MyDrive/ipynb stuff/Madden_Ratings.csv")

FUZZY_SAME_TEAM_THRESHOLD = 83.4
FUZZY_ANY_TEAM_THRESHOLD  = 85

In [60]:
# Madden uses verbose position names; rosters uses abbreviations
position_mapping = {
    "Quarterback":    "QB", "Halfback":       "RB", "Fullback":     "FB",
    "Wide Receiver":  "WR", "Tight End":      "TE",
    "Left Tackle":    "OT", "Right Tackle":   "OT",
    "Left Guard":     "G",  "Right Guard":    "G",  "Center":       "C",
    "Left Edge":      "DE", "Right Edge":     "DE",
    "Defensive Tackle": "DT",
    "Mike Backer":    "MLB",
    "Sam Backer":     "OLB", "Weak Backer":   "OLB",
    "Cornerback":     "CB", "Free Safety":    "FS", "Strong Safety": "S",
    "Kicker":         "K",  "Punter":         "P",  "Long Snapper":  "LS",
}

# Madden uses full team names; rosters uses abbreviations.
# Note: Madden uses "NY Giants" / "NY Jets" — not the full city names.
team_mapping = {
    "Arizona Cardinals":    "ARI", "Atlanta Falcons":      "ATL",
    "Baltimore Ravens":     "BAL", "Buffalo Bills":        "BUF",
    "Carolina Panthers":    "CAR", "Chicago Bears":        "CHI",
    "Cincinnati Bengals":   "CIN", "Cleveland Browns":     "CLE",
    "Dallas Cowboys":       "DAL", "Denver Broncos":       "DEN",
    "Detroit Lions":        "DET", "Green Bay Packers":    "GB",
    "Houston Texans":       "HOU", "Indianapolis Colts":   "IND",
    "Jacksonville Jaguars": "JAX", "Kansas City Chiefs":   "KC",
    "Las Vegas Raiders":    "LV",  "Los Angeles Chargers": "LAC",
    "Los Angeles Rams":     "LA",  "Miami Dolphins":       "MIA",
    "Minnesota Vikings":    "MIN", "New England Patriots": "NE",
    "New Orleans Saints":   "NO",  "NY Giants":            "NYG",
    "NY Jets":              "NYJ", "Philadelphia Eagles":  "PHI",
    "Pittsburgh Steelers":  "PIT", "San Francisco 49ers":  "SF",
    "Seattle Seahawks":     "SEA", "Tampa Bay Buccaneers": "TB",
    "Tennessee Titans":     "TEN", "Washington Commanders":"WAS",
}

# Players whose fuzzy matches were manually reviewed and confirmed correct.
# These are nickname / encoding variants — e.g. Patrick → Pat, Da'Ron → Daron.
# Only matches in this set will receive a gsis_id from the fuzzy pass.
VERIFIED_FUZZY_MATCHES = {
    "Kenneth Murray",    "Zah Frazier",          "Tre' Harris",
    "Lajohntay Wester",  "Audric Estime",         "Joshua Palmer",
    "Chris Roland-Wallace", "Cor'Dale Flott",     "Matt Orzech",
    "Andres Borregales", "Da'Ron Payne",          "Drew Ogletree",
    "Foyesade Oluokun",  "Dru Phillips",          "Patrick Surtain",
    "Kamren Curl",       "Dax Hill",              "Olusegun Oluwatimi",
    "JuJu Brents",       "Benjamin Yurosek",      "Michael Jordan",
}

In [61]:
#Name normalization

def normalize_name(name: str) -> str:
    """
    Standardizes player names to reduce merge failures caused by formatting
    differences between Madden and nflverse roster data.

    Handles:
      - Suffixes     : Jr., Sr., II, III, IV, V  →  stripped
      - Initial dots : D.K. → DK,  A.J. → AJ
      - Apostrophes  : curly/backtick variants    →  straight apostrophe
      - Whitespace   : leading/trailing spaces    →  stripped
    """
    if pd.isna(name):
        return name
    name = str(name).strip()
    name = re.sub(r'[''`]', "'", name)                                               # normalize apostrophes
    name = re.sub(r'\s+(Jr\.?|Sr\.?|II|III|IV|V)$', '', name, flags=re.IGNORECASE)  # strip suffixes
    name = re.sub(r'(?<=[A-Z])\.(?=[A-Z]\.?|\s|$)', '', name)                       # strip initial periods
    return name.strip()


In [62]:
# Clean madden df

madden = madden2026.copy()

original_teams  = madden["Team"].copy()                          # preserve for error reporting
madden["Name"]  = (madden["First Name"] + " " + madden["Last Name"]).apply(normalize_name)
madden["position"] = madden["Position"].map(position_mapping)
madden["Team"]  = madden["Team"].map(team_mapping)
madden = madden.drop(columns=["First Name", "Last Name", "Position"])

# Reorder columns: identifiers first, then all Madden attributes
front_cols = ["Name", "position", "Team"]
madden     = madden[front_cols + [c for c in madden.columns if c not in front_cols]]

# Guard: any team that didn't map means team_mapping is incomplete
unmapped_teams = original_teams[madden["Team"].isna()].unique()
assert len(unmapped_teams) == 0, \
    f"[!] Unmapped teams — add to team_mapping: {unmapped_teams}"

madden["Name_norm"] = madden["Name"]   # name is already normalized; alias for merge key

#clean rosters df

rosters_clean = (
    rosters[["full_name", "gsis_id", "team"]]
    .copy()
    .assign(Name_norm=lambda df: df["full_name"].apply(normalize_name))
    .drop_duplicates(subset=["Name_norm", "team"])
)

print(f"Madden cleaned:  {len(madden):,} players")
print(f"Rosters cleaned: {len(rosters_clean):,} players")

Madden cleaned:  2,035 players
Rosters cleaned: 3,094 players


In [63]:

#Waterfall merge

# Merges from most to least specific, tracking residuals at every step.
#   Pass 1  —  Name + Team         (precise; handles same-name players on different teams)
#   Pass 2  —  Name only           (unambiguous only — one gsis_id per name in rosters)
#   Pass 3  —  Name only           (last resort — takes first match; flags for review)

def split_merge(df: pd.DataFrame):
    """
    Splits a post-merge dataframe into matched and unmatched rows.
    Strips roster join columns from unmatched rows to keep the schema clean.
    """
    matched   = df[df["gsis_id"].notna()].copy()
    unmatched = df[df["gsis_id"].isna()].drop(columns=["gsis_id", "full_name", "team"],
                                               errors="ignore")
    return matched, unmatched


# Pass 1: Name + Team
p1 = madden.merge(
    rosters_clean,
    left_on=["Name_norm", "Team"],
    right_on=["Name_norm", "team"],
    how="left"
)
matched_p1, unmatched_p1 = split_merge(p1)

# Pass 2: Name only — unambiguous matches
# Restrict to names where exactly one gsis_id exists across all rosters.
# Prevents incorrect matches when two players share the same normalized name.
name_uid_counts   = rosters_clean.groupby("Name_norm")["gsis_id"].nunique()
unambiguous_names = name_uid_counts[name_uid_counts == 1].index
rosters_unamb     = rosters_clean[rosters_clean["Name_norm"].isin(unambiguous_names)]

p2 = unmatched_p1.merge(
    rosters_unamb[["Name_norm", "gsis_id", "full_name", "team"]],
    on="Name_norm",
    how="left"
)
matched_p2, unmatched_p2 = split_merge(p2)

# Pass 3: Name only — last resort
# No ambiguity filter; accepts first match. Risky — fuzzy pass will catch errors.
p3 = unmatched_p2.merge(
    rosters_clean[["Name_norm", "gsis_id", "full_name", "team"]],
    on="Name_norm",
    how="left"
)
p3 = p3.drop_duplicates(subset=["Name"], keep="first")
matched_p3, still_missing = split_merge(p3)

print(f"[✓] Pass 1  (Name + Team):        {len(matched_p1):>4}  ({len(matched_p1)/len(madden):.1%})")
print(f"[✓] Pass 2  (Name, unambiguous):  {len(matched_p2):>4}")
print(f"[✓] Pass 3  (Name, last resort):  {len(matched_p3):>4}")
print(f"[→] Entering fuzzy match:         {len(still_missing):>4} remaining")

[✓] Pass 1  (Name + Team):        1591  (78.2%)
[✓] Pass 2  (Name, unambiguous):   383
[✓] Pass 3  (Name, last resort):     3
[→] Entering fuzzy match:           59 remaining


In [67]:
#Fuzzy Matching for Residuals

# Catches nickname variants (Patrick → Pat), encoding differences (accents,
# apostrophes), and minor spelling variations that exact merges cannot handle.
#
# Strategy:
#   1. Same-team match first  (lower threshold — team filter reduces false positives)
#   2. Any-team fallback       (higher threshold — no team context means more risk)

roster_name_list = rosters_clean["Name_norm"].tolist()

def fuzzy_match_player(
    madden_name: str,
    madden_team: str,
    rosters_df: pd.DataFrame,
    all_roster_names: list,
) -> tuple:
    """
    Finds the best fuzzy name match for a single unmatched player.

    Args:
        madden_name:      Normalized player name from Madden.
        madden_team:      Team abbreviation from Madden.
        rosters_df:       Cleaned rosters dataframe.
        all_roster_names: Precomputed list of all roster Name_norm values.

    Returns:
        Tuple of (matched_name, score, gsis_id). Returns (None, 0, None) if
        no match clears the threshold.
    """
    same_team = rosters_df[rosters_df["team"] == madden_team]

    # Attempt 1: same-team match
    if len(same_team) > 0:
        result = process.extractOne(
            madden_name, same_team["Name_norm"].tolist(), scorer=fuzz.token_sort_ratio
        )
        if result and result[1] >= FUZZY_SAME_TEAM_THRESHOLD:
            gsis = same_team[same_team["Name_norm"] == result[0]]["gsis_id"].values
            return result[0], result[1], (gsis[0] if len(gsis) > 0 else None)

    # Attempt 2: any-team match (compensate with higher threshold)
    result = process.extractOne(madden_name, all_roster_names, scorer=fuzz.token_sort_ratio)
    if result and result[1] >= FUZZY_ANY_TEAM_THRESHOLD:
        gsis = rosters_df[rosters_df["Name_norm"] == result[0]]["gsis_id"].values
        return result[0], result[1], (gsis[0] if len(gsis) > 0 else None)

    return None, 0, None


fuzzy_results = [
    {
        "Name":       row["Name"],
        "Team":       row["Team"],
        "Matched To": matched_to,
        "Score":      score,
        "gsis_id":    gsis_id,
    }
    for _, row in still_missing.iterrows()
    for matched_to, score, gsis_id in [
        fuzzy_match_player(row["Name"], row["Team"], rosters_clean, roster_name_list)
    ]
]

fuzzy_df = pd.DataFrame(fuzzy_results).sort_values("Score", ascending=False)
print(fuzzy_df.to_string())


                    Name Team                Matched To      Score     gsis_id
48        Kenneth Murray  DAL           Kenneth Murray,  96.551724  00-0036441
40           Zah Frazier  CHI              Zach Frazier  95.652174  00-0039896
30           Tre' Harris  LAC                Tre Harris  95.238095  00-0040727
37      Lajohntay Wester  BAL          LaJohntay Wester  93.750000  00-0040075
35         Audric Estime   NO             Audric Estimé  92.307692  00-0039373
19         Joshua Palmer  BUF               Josh Palmer  91.666667  00-0036988
43  Chris Roland-Wallace   KC  Christian Roland-Wallace  90.909091  00-0039324
11        Cor'Dale Flott  NYG             Cordale Flott  88.888889  00-0037758
55           Matt Orzech   GB            Matthew Orzech  88.000000  00-0035118
26     Andres Borregales   NE           Andy Borregales  87.500000  00-0040200
5           Da'Ron Payne  WAS               Daron Payne  86.956522  00-0034333
18      Darious Williams   LA            Mario Willi

In [68]:

# Apply Verified Fuzzy Matches

# Fuzzy matching can produce false positives (e.g. "Logan Wilson" matching
# "Roman Wilson"). Only matches in VERIFIED_FUZZY_MATCHES — confirmed correct
# via manual review — are accepted. All others remain in the dataset with
# gsis_id = NaN. A Madden rating is still analytically valid without a
# roster match.

verified_gsis = (
    fuzzy_df[
        fuzzy_df["Name"].isin(VERIFIED_FUZZY_MATCHES) &
        fuzzy_df["gsis_id"].notna()
    ]
    [["Name", "gsis_id"]]
)

# Merge verified gsis_ids back into all still-missing rows
matched_fuzzy = still_missing.merge(verified_gsis, on="Name", how="left")

accepted  = matched_fuzzy["gsis_id"].notna().sum()
remaining = matched_fuzzy["gsis_id"].isna().sum()
print(f"[✓] Fuzzy matches accepted:  {accepted}")
print(f"[→] No gsis_id (kept):       {remaining}  — Madden rating still valid")


[✓] Fuzzy matches accepted:  18
[→] No gsis_id (kept):       41  — Madden rating still valid


In [69]:
# Final Assembly & Validation

# Combine all passes. Sort by Overall descending before deduplication so that
# when a player appears twice in Madden (e.g. traded mid-year), the higher-rated
# entry is kept.
all_matched = pd.concat([matched_p1, matched_p2, matched_p3, matched_fuzzy],
                        ignore_index=True)
all_matched = all_matched.drop(columns=["Name_norm", "full_name", "team"], errors="ignore")
all_matched = all_matched.sort_values("Overall", ascending=False)

# Deduplicate on gsis_id — but only where gsis_id is not null.
# Deduplicating on NaN would incorrectly collapse all unmatched players into one row.
has_gsis    = all_matched[all_matched["gsis_id"].notna()].drop_duplicates(subset=["gsis_id"], keep="first")
no_gsis     = all_matched[all_matched["gsis_id"].isna()]
madden_final = pd.concat([has_gsis, no_gsis], ignore_index=True) \
                 .sort_values("Overall", ascending=False)          \
                 .reset_index(drop=True)

# ── Validation Report ─────────────────────────────────────────────────────────
total        = len(madden_final)
with_gsis    = madden_final["gsis_id"].notna().sum()
without_gsis = madden_final["gsis_id"].isna().sum()

print("═" * 52)
print("  FINAL DATASET SUMMARY")
print("═" * 52)
print(f"  Total players:             {total:>5,}")
print(f"  Matched to roster:         {with_gsis:>5,}  ({with_gsis/total:.1%})")
print(f"  No gsis_id (valid):        {without_gsis:>5,}  ({without_gsis/total:.1%})")
print("═" * 52)
print(f"  Breakdown by pass:")
print(f"    Pass 1  (Name + Team):         {len(matched_p1):>4}")
print(f"    Pass 2  (Name, unambiguous):   {len(matched_p2):>4}")
print(f"    Pass 3  (Name, last resort):   {len(matched_p3):>4}")
print(f"    Fuzzy   (verified):            {accepted:>4}")
print(f"    No match (null gsis_id):       {without_gsis:>4}")
print("═" * 52)

════════════════════════════════════════════════════
  FINAL DATASET SUMMARY
════════════════════════════════════════════════════
  Total players:             2,032
  Matched to roster:         1,991  (98.0%)
  No gsis_id (valid):           41  (2.0%)
════════════════════════════════════════════════════
  Breakdown by pass:
    Pass 1  (Name + Team):         1591
    Pass 2  (Name, unambiguous):    383
    Pass 3  (Name, last resort):      3
    Fuzzy   (verified):              18
    No match (null gsis_id):         41
════════════════════════════════════════════════════


In [71]:
madden = madden_final.copy()
madden.head(10)

,Name,position,Team,Overall,Player Image,Team Logo,ACCELERATION,AGILITY,JUMPING,STAMINA,STRENGTH,AWARENESS,BCVISION,BLOCKSHEDDING,BREAKSACK,BREAKTACKLE,CARRYING,CATCHINTRAFFIC,CATCHING,CHANGEOFDIRECTION,DEEPROUTERUNNING,FINESSEMOVES,HITPOWER,IMPACTBLOCKING,INJURY,JUKEMOVE,KICKACCURACY,KICKPOWER,KICKRETURN,LEADBLOCK,MANCOVERAGE,MEDIUMROUTERUNNING,OVERALL,PASSBLOCK,PASSBLOCKFINESSE,PASSBLOCKPOWER,PLAYACTION,PLAYRECOGNITION,POWERMOVES,PRESS,PURSUIT,RELEASE,RUNBLOCK,RUNBLOCKFINESSE,RUNBLOCKPOWER,RUNNINGSTYLE,SHORTROUTERUNNING,SPECTACULARCATCH,SPEED,SPINMOVE,STIFFARM,TACKLE,THROWACCURACYDEEP,THROWACCURACYMID,THROWACCURACYSHORT,THROWONTHERUN,THROWPOWER,THROWUNDERPRESSURE,TOUGHNESS,TRUCKING,ZONECOVERAGE,gsis_id
0,Ja'Marr Chase,WR,CIN,99,https://ratings-images-prod.pulse.ea.com/madde...,https://drop-assets.ea.com/images/5oemvWKqCFgB...,94,92,98,96,73,99,98,37,29,88,71,94,98,97,98,10,27,46,92,90,17,24,60,27,13,97,99,37,20,26,10,17,10,15,36,99,53,27,31,Default,96,97,95,83,76,39,10,15,20,19,33,22,88,71,17,00-0036900
1,Myles Garrett,DE,CLE,99,https://ratings-images-prod.pulse.ea.com/madde...,https://drop-assets.ea.com/images/4qbblHggBjaH...,90,86,94,89,96,97,39,96,12,34,55,21,52,71,4,93,84,90,92,36,20,20,10,17,51,8,99,45,45,45,10,94,99,44,97,31,45,45,45,Long Stride Default,15,25,87,40,48,92,27,30,31,23,55,16,87,36,51,00-0033868
2,Lane Johnson,OT,PHI,99,https://ratings-images-prod.pulse.ea.com/madde...,https://drop-assets.ea.com/images/1Nt5pFCpjcBc...,78,75,82,88,93,99,60,56,39,30,67,58,64,58,25,60,60,95,84,54,22,25,10,96,12,33,99,98,94,98,39,29,53,10,55,45,97,96,98,Default,46,57,77,50,55,60,50,54,59,52,65,41,91,65,20,00-0030561
3,Josh Allen,QB,BUF,99,https://ratings-images-prod.pulse.ea.com/madde...,https://drop-assets.ea.com/images/212FXcpkDUjs...,91,85,92,98,81,97,97,23,98,86,63,20,40,82,13,9,11,26,99,81,14,22,13,17,12,15,99,14,10,15,97,15,9,9,33,18,17,10,10,Default Stride Bread Loaf,22,23,88,78,83,32,90,88,97,99,98,94,98,76,18,00-0034857
4,Christian Gonzalez,CB,NE,98,https://ratings-images-prod.pulse.ea.com/madde...,https://drop-assets.ea.com/images/6Z0rWjQfkWgv...,93,94,95,90,59,98,75,53,10,67,60,62,74,95,20,37,49,48,90,74,42,46,55,15,98,27,98,30,24,25,6,98,28,86,77,40,32,27,32,Default Stride Loose,37,73,96,66,57,76,6,10,15,10,35,10,78,42,90,00-0039147
5,Micah Parsons,DE,GB,98,https://ratings-images-prod.pulse.ea.com/madde...,https://drop-assets.ea.com/images/4ztQk5Xogdkr...,95,90,85,90,91,97,71,80,12,57,69,44,62,77,31,97,84,84,93,71,20,20,60,20,59,36,98,45,45,45,6,95,92,57,94,36,45,45,45,Default Stride Loose,40,47,90,61,58,89,6,6,6,6,30,15,87,62,64,00-0036932
6,George Kittle,TE,SF,98,https://ratings-images-prod.pulse.ea.com/madde...,https://drop-assets.ea.com/images/1SoIzWJtk28y...,87,83,90,94,82,99,92,31,18,78,75,89,95,77,77,18,38,78,88,82,22,19,31,65,15,79,98,62,58,62,10,35,20,20,36,83,77,71,76,Default Stride Loose,84,89,87,64,83,34,10,15,20,10,30,14,98,84,24,00-0033288
7,Penei Sewell,OT,DET,98,https://ratings-images-prod.pulse.ea.com/madde...,https://drop-assets.ea.com/images/7cuE4tZcli1u...,74,60,70,95,97,98,20,35,13,15,35,17,44,58,5,16,33,96,94,10,20,23,10,97,12,17,98,90,86,94,6,19,24,10,37,15,99,99,99,Default Stride Awkward,28,15,73,10,25,32,6,6,6,6,18,14,91,30,12,00-0036880
8,Matthew Stafford,QB,LA,98,https://ratings-images-prod.pulse.ea.com/madde...,https://drop-assets.ea.com/images/wS8JnLBEBj9e...,70,71,75,91,66,99,71,26,73,52,62,15,26,66,12,10,12,27,87,55,25,28,10,17,10,14,98,25,22,24,90,17,10,10,29,15,22,19,17,Default,15,15,78,39,36,28,95,95,98,88,95,84,98,47,22,00-0026498
9,Jahmyr Gibbs,RB,DET,98,https://ratings-images-prod.pulse.ea.com/madde...,https://drop-assets.ea.com/images/7cuE4tZcli1u...,95,93,81,85,72,88,92,30,25,84,96,64,74,92,63,10,28,32,90,96,10,14,80,29,12,69,98,51,29,34,10,16,10,16,37,65,30,24,16,Default,73,64,98,94,72,30,13,17,21,15,26,14,84,82,15,00-0039139


## Analysis